# L06 · 실제 train/eval

## Goal

**예상 시간:** 50분 · **경로:** full

- 하드웨어를 사전 점검한다
- pinned preset을 읽는다
- 안전한 실패를 해석한다

### 현재 위치: L05 → **L06** → L07

```text
Prompt/Data -> state source -> ... -> L06 -> ... -> fair evaluation
```

Alt text: The course map highlights L06 between its prerequisite and next lesson; every method remains connected to the same evaluation stage.

## Setup

In [1]:
LESSON_ID = "L06"
from pathlib import Path
import sys
import torch

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = Path.cwd().parents[1]
sys.path.insert(0, str(repo_root / "src"))

import opd_study
from opd_study.device import resolve_device
from opd_study.utils import seed_everything

seed_everything(42)
device_report = resolve_device("cpu")
print({"lesson": LESSON_ID, "opd_study": opd_study.__version__,
       "torch": torch.__version__, "device": device_report.selected,
       "profile": "toy", "network": "not required"})

{'lesson': 'L06', 'opd_study': '0.1.0.dev0', 'torch': '2.13.0', 'device': 'cpu', 'profile': 'toy', 'network': 'not required'}


## Steps

### 1/3 · 8–12 min

실제 모델 경로는 다운로드 전에 revision·라이선스·크기·device를 보여준다. 이 노트북은 offline toy fallback을 기본으로 하며 네트워크 결과를 꾸며내지 않는다.

그림 대체 설명: 출력의 label과 숫자는 색 없이도 읽을 수 있다.

### 핵심 원리

실제 학습 경로는 실행보다 **preflight**가 먼저다. dataset/model revision, 예상 byte, license 동의, tokenizer·chat template, dtype, device와 fallback 정책을 확인한 뒤에만 다운로드한다. official test는 학습에 쓰지 않고 train에서 validation을 고정 seed로 분리한다.

LoRA는 frozen weight에 작은 low-rank update `B·A`를 학습한다. QLoRA는 base weight까지 4-bit로 양자화해 VRAM을 더 줄이지만 bitsandbytes/CUDA 조합을 실제 forward-backward-save-reload로 검증해야 한다. 이 레포는 실패 시 full fine-tuning으로 조용히 바꾸지 않는다.

### 실제 구현: 왜 이렇게 만들었나

research config는 `backend=research`, exact 40-char revision, download byte와 동의 flag를 가진다. preflight는 package 존재뿐 아니라 실제 import, PyTorch version, device와 QLoRA capability까지 검사한다. 모델 loader는 vocab와 chat template가 다르면 즉시 중단한다.

실제 코드: [`preflight.py`](../../src/opd_study/research/preflight.py), [`hf_backend.py`](../../src/opd_study/research/hf_backend.py).

In [2]:
import inspect
from opd_study.research import research_preflight

objects_to_show = (research_preflight,)
for object_to_show in objects_to_show:
    source_lines = inspect.getsource(object_to_show).splitlines()
    print(f"\n# {object_to_show.__module__}.{object_to_show.__qualname__}")
    print("\n".join(source_lines[:80]))
    if len(source_lines) > 80:
        print(f"... {len(source_lines) - 80} more lines; open the linked source file")


# opd_study.research.preflight.research_preflight
def research_preflight(config: ExperimentConfig) -> ResearchPreflight:
    if config.backend != "research":
        raise ValueError("research preflight requires backend=research")
    required = ["transformers", "datasets", "accelerate", "peft"]
    if config.model.finetuning == "qlora":
        required.append("bitsandbytes")
    missing = tuple(name for name in required if importlib.util.find_spec(name) is None)
    blockers: list[str] = []
    if missing:
        blockers.append("missing optional packages: " + ", ".join(missing))
    package_versions: dict[str, str] = {}
    for distribution in ("torch", *required):
        try:
            package_versions[distribution] = version(distribution)
        except PackageNotFoundError:
            continue
    if Version(package_versions["torch"]) < Version("2.2"):
        blockers.append(
            f"research backend requires torch>=2.2; found {package_versions['torch']}"
        )
 

### 다른 선택지는 없나?

full fine-tuning은 가장 직접적이나 메모리가 크다. LoRA는 넓은 환경에서 안정적이고, QLoRA는 VRAM을 줄이는 대신 CUDA/bitsandbytes 의존성이 커진다. macOS에서는 QLoRA를 full-FT로 대체하지 말고 LoRA/CPU toy 또는 검증된 CUDA를 선택한다.

### 2/3 · 실행하고 관찰하기

실행 전 예측: L06의 첫 출력에서 가장 먼저 확인해야 할 invariant는 무엇일까? 한 문장으로 적고 실행한다.

In [3]:
from opd_study.config import load_config
from opd_study.device import require_qlora

toy = load_config(repo_root / "configs/toy/default.yaml")
laptop = load_config(repo_root / "configs/laptop/gsm8k_lora.yaml")
qlora = load_config(repo_root / "configs/laptop/gsm8k_qlora.yaml")
print("toy:", toy.profile, toy.backend, toy.data.id)
print("laptop pins:", laptop.model.student, laptop.model.student_revision)
print("CUDA preset:", qlora.model.finetuning, qlora.training.device, qlora.training.precision)
print("download estimate is documented before any network call:", laptop.data.expected_download_bytes)

toy: toy mini tiny_arithmetic
laptop pins: Qwen/Qwen3-0.6B c1899de289a04d12100db370d81485cdf75e47ca
CUDA preset: qlora cuda float16
download estimate is documented before any network call: 2725633


In [4]:
try:
    require_qlora(device_report)
except RuntimeError as error:
    print("safe QLoRA block:", error)

RUN_OPTIONAL_NETWORK = False
print("Qwen/GSM8K smoke enabled:", RUN_OPTIONAL_NETWORK)
print("Exact command: opd-study research-train --config configs/laptop/gsm8k_lora.yaml --smoke --accept-dataset-license --accept-model-license")

safe QLoRA block: QLoRA requires a validated NVIDIA CUDA + bitsandbytes environment in this project; macOS/MPS and CPU never fall back to full fine-tuning
Qwen/GSM8K smoke enabled: False
Exact command: opd-study research-train --config configs/laptop/gsm8k_lora.yaml --smoke --accept-dataset-license --accept-model-license


## Checks

In [5]:
assert len(laptop.model.student_revision) == 40
assert laptop.model.trust_remote_code is False
assert qlora.model.finetuning == "qlora" and qlora.training.device == "cuda"
assert RUN_OPTIONAL_NETWORK is False
print("check passed: pinned assets, no remote code, unsupported QLoRA does not fall back")

check passed: pinned assets, no remote code, unsupported QLoRA does not fall back


**연습 (8분):** laptop config에서 device를 `mps`, finetuning을 `qlora`로 가정한 실패 보고서를 작성하라. fallback으로 full FT를 제안하면 안 된다.

<details><summary>확인 기준</summary>bitsandbytes/CUDA 제약, 예상 다운로드, 동의 flag와 LoRA/CPU toy 대안을 명시한다.</details>

## 내가 자주 틀리는 것

### M1 — 모델 이름만 pin하고 revision은 최신으로 두기

- 틀린 형태: Hub ID만 기록해 재실행 때 weight가 바뀐다.
- 왜 틀렸나: code/config/weight가 이동할 수 있다.
- 고친 형태: 40-char revision, license, bytes와 checksum을 기록한다.
- 관련 검사: `test_all_checked_in_presets_parse`

### M2 — QLoRA 실패를 full FT로 숨기기

- 틀린 형태: bitsandbytes가 안 되면 더 큰 메모리 경로로 자동 전환한다.
- 왜 틀렸나: OOM과 결과 의미 변경을 숨긴다.
- 고친 형태: 명시적으로 차단하고 LoRA 또는 CUDA 환경을 선택하게 한다.
- 관련 검사: `test_qlora_preset_is_explicit_cuda_and_opt_in`

## 60초 요약

1. 하드웨어를 사전 점검한다
2. pinned preset을 읽는다
3. 안전한 실패를 해석한다

## Next Steps

다음 노트북으로 가기 전, 위 assertion을 다시 실행하고 틀린 예측 한 줄을 남긴다.

### Sources

- [`gkd`](https://arxiv.org/abs/2306.13649v3) · `2306.13649v3` · license `CC-BY-4.0` · [audited manifest](../../docs/sources.yml)
- [`openai/gsm8k`](https://huggingface.co/datasets/openai/gsm8k) · `740312add88f781978c0658806c59bc2815b9866` · license `MIT` · [audited manifest](../../docs/sources.yml)
- [`Qwen/Qwen3-0.6B`](https://huggingface.co/Qwen/Qwen3-0.6B) · `c1899de289a04d12100db370d81485cdf75e47ca` · license `Apache-2.0` · [audited manifest](../../docs/sources.yml)
- [`Qwen/Qwen3-1.7B`](https://huggingface.co/Qwen/Qwen3-1.7B) · `70d244cc86ccca08cf5af4e1e306ecf908b1ad5e` · license `Apache-2.0` · [audited manifest](../../docs/sources.yml)